# RoBERTa fine-tuned LLM-text detector — Colab notebook

**Purpose.** Train a RoBERTa-base classifier on P0 (Wikipedia human vs Llama-3.1-8B), then evaluate on all paraphrase stages (P1/P2, standard/simplified). Outputs a `roberta_predictions.csv` that integrates back into the local multi-seed pipeline via `src/experiments/integrate_neural_predictions.py`.

**Why a Colab notebook.** The local pipeline runs on Windows with CPU-only torch. RoBERTa fine-tuning needs a GPU; Colab's free T4 is sufficient (~3 min/seed total).

## How to run

1. Open [colab.research.google.com](https://colab.research.google.com).
2. File → Upload notebook → choose `colab/roberta_detector_finetune.ipynb`.
3. Runtime → Change runtime type → **GPU (T4)**.
4. Run cell 1 to install dependencies (~1 min).
5. Run cell 3, click *Choose Files* and upload **all five JSONL files** at once: `p0.jsonl`, `p1_test.jsonl`, `p2_test.jsonl`, `p1_test_simplified.jsonl`, `p2_test_simplified.jsonl`, plus the three split files: `train_ids.txt`, `val_ids.txt`, `test_ids.txt`.
6. Run the remaining cells in order. Training takes ~3 min/seed on T4.
7. Cell 9 downloads `roberta_predictions.csv` to your local Downloads folder.
8. Move that CSV into your local repo at `results/eval/roberta_predictions.csv` and run `python src/experiments/integrate_neural_predictions.py` to fold it into the multi-seed analysis.

## Files this notebook expects

| Where it comes from | File | Purpose |
|---|---|---|
| `data/p0/p0.jsonl` | `p0.jsonl` | 1000 P0 rows |
| `data/p1/p1_test.jsonl` | `p1_test.jsonl` | 100 paraphrased rows (standard) |
| `data/p2/p2_test.jsonl` | `p2_test.jsonl` | 100 doubly paraphrased rows |
| `data/p1/p1_test_simplified.jsonl` | `p1_test_simplified.jsonl` | 100 rows, simplified |
| `data/p2/p2_test_simplified.jsonl` | `p2_test_simplified.jsonl` | 100 rows, simplified |
| `data/splits/train_ids.txt` | `train_ids.txt` | 800 IDs |
| `data/splits/val_ids.txt` | `val_ids.txt` | 100 IDs |
| `data/splits/test_ids.txt` | `test_ids.txt` | 100 IDs |


## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas
import torch, transformers, datasets
print('torch:', torch.__version__, ' CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('transformers:', transformers.__version__)
print('datasets:', datasets.__version__)

## 2. Upload data files

When the file picker appears, select **all eight files at once**: the five `.jsonl` files and the three `_ids.txt` files. Files land in the Colab working directory.

In [ ]:
from google.colab import files
uploaded = files.upload()
import os
print('\nFiles in working dir:')
for f in sorted(os.listdir('.')):
    if f.endswith('.jsonl') or f.endswith('.txt'):
        print(f'  {f}  ({os.path.getsize(f)} bytes)')

## 3. Load data into HuggingFace `Dataset` objects

In [ ]:
import json
from datasets import Dataset

def load_jsonl(p):
    with open(p, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def load_ids(p):
    return set(open(p, 'r', encoding='utf-8').read().splitlines())

p0 = load_jsonl('p0.jsonl')
train_ids = load_ids('train_ids.txt')
val_ids   = load_ids('val_ids.txt')
test_ids  = load_ids('test_ids.txt')

def split_rows(rows, ids):
    return [r for r in rows if r['id'] in ids]

train_rows = split_rows(p0, train_ids)
val_rows   = split_rows(p0, val_ids)
test_rows  = split_rows(p0, test_ids)

p1_std = load_jsonl('p1_test.jsonl')
p2_std = load_jsonl('p2_test.jsonl')
p1_sim = load_jsonl('p1_test_simplified.jsonl')
p2_sim = load_jsonl('p2_test_simplified.jsonl')

def rows_to_dataset(rows):
    return Dataset.from_dict({
        'id':    [r['id'] for r in rows],
        'text':  [r['text'] for r in rows],
        'label': [1 if r['label'] == 'llm' else 0 for r in rows],
    })

ds = {
    'train': rows_to_dataset(train_rows),
    'val':   rows_to_dataset(val_rows),
    'P0_test':            rows_to_dataset(test_rows),
    'P1_test_standard':   rows_to_dataset(p1_std),
    'P2_test_standard':   rows_to_dataset(p2_std),
    'P1_test_simplified': rows_to_dataset(p1_sim),
    'P2_test_simplified': rows_to_dataset(p2_sim),
}
for k, v in ds.items():
    print(f'  {k}: n={len(v)}  label-mean={sum(v["label"])/len(v):.3f}')

## 4. Tokenize with RoBERTa tokenizer

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'roberta-base'   # ~125M params; on T4 fine-tunes in ~3 min total
MAX_LEN    = 256              # paragraphs are 100-200 words; 256 tokens fits with margin

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tok(batch['text'], padding=False, truncation=True, max_length=MAX_LEN)

ds_tok = {k: v.map(tokenize_fn, batched=True) for k, v in ds.items()}
for k, v in ds_tok.items():
    print(f'  {k}: tokenized to columns: {v.column_names}')

## 5. Fine-tune RoBERTa (multi-seed for stability)

Trains 3 seeds. For each seed, the model is initialized identically (same pre-trained weights) but uses a different RNG for data shuffling, dropout, and classifier-head initialisation. Across-seed variance gives an honest CI on the neural-detector numbers.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification, TrainingArguments, Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, matthews_corrcoef
import numpy as np, pandas as pd, os

SEEDS    = [42, 0, 1]            # 3 seeds; on T4 each takes ~2 min for 4 epochs
EPOCHS   = 4
BATCH    = 16
LR       = 2e-5
WD       = 0.01
WARMUP   = 0.1

data_collator = DataCollatorWithPadding(tokenizer=tok)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = probs >= 0.5
    p, r, f, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(labels, probs) if len(set(labels)) > 1 else float('nan')
    except ValueError:
        auc = float('nan')
    return {
        'acc':       accuracy_score(labels, preds),
        'precision': p, 'recall': r, 'f1': f, 'auroc': auc,
        'mcc':       matthews_corrcoef(labels, preds),
    }

splits_eval = ['P0_test', 'P1_test_standard', 'P2_test_standard',
               'P1_test_simplified', 'P2_test_simplified']

all_pred_rows = []
for seed in SEEDS:
    print(f'\n========== SEED {seed} ==========')
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = TrainingArguments(
        output_dir=f'./roberta_seed{seed}',
        eval_strategy='epoch',
        save_strategy='no',
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH * 2,
        num_train_epochs=EPOCHS,
        weight_decay=WD,
        warmup_steps=int(WARMUP * (len(ds_tok["train"]) // BATCH)),
        logging_steps=25,
        seed=seed,
        data_seed=seed,
        report_to='none',
        fp16=True,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok['train'],
        eval_dataset=ds_tok['val'],
        processing_class=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()

    print(f'\n-- seed {seed} predictions on each paraphrase split --')
    for sp_name in splits_eval:
        out = trainer.predict(ds_tok[sp_name])
        probs = torch.softmax(torch.tensor(out.predictions), dim=-1)[:, 1].numpy()
        preds = (probs >= 0.5).astype(int)
        ids   = ds_tok[sp_name]['id']
        labels = ds_tok[sp_name]['label']
        for i, pid in enumerate(ids):
            all_pred_rows.append({
                'seed': seed,
                'detector': 'roberta_base',
                'split': sp_name,
                'id':     pid,
                'y_true': int(labels[i]),
                'y_prob': float(probs[i]),
                'y_pred': int(preds[i]),
            })
        m = compute_metrics((out.predictions, np.array(labels)))
        print(f'  {sp_name:<25}  acc={m["acc"]:.4f}  f1={m["f1"]:.4f}  auroc={m["auroc"]:.4f}')

df_preds = pd.DataFrame(all_pred_rows)
df_preds.to_csv('roberta_predictions.csv', index=False)
print(f'\nSaved roberta_predictions.csv with {len(df_preds)} rows')
df_preds.head()

## 6. Quick summary across seeds

In [ ]:
import numpy as np

rows = []
for sp in splits_eval:
    for seed in SEEDS:
        sub = df_preds[(df_preds.split == sp) & (df_preds.seed == seed)]
        rows.append({
            'split': sp, 'seed': seed,
            'acc':   accuracy_score(sub.y_true, sub.y_pred),
            'f1':    precision_recall_fscore_support(sub.y_true, sub.y_pred, average='binary', zero_division=0)[2],
            'auroc': roc_auc_score(sub.y_true, sub.y_prob) if len(set(sub.y_true)) > 1 else float('nan'),
        })
summary = pd.DataFrame(rows)
agg = summary.groupby('split').agg({
    'acc':   ['mean', 'std'],
    'f1':    ['mean', 'std'],
    'auroc': ['mean', 'std'],
})
print('Across-seed RoBERTa-base performance:\n')
print(agg)

## 7. Download artifacts

Move the downloaded `roberta_predictions.csv` to your local repo at:

    Analysis-of-Human-vs.-LLM-Generated-Text/results/eval/roberta_predictions.csv

Then run locally:

    python src/experiments/integrate_neural_predictions.py

to fold these predictions into the multi-seed bootstrap / hardness / AURC analysis.

In [ ]:
from google.colab import files
files.download('roberta_predictions.csv')

## (Optional) Save the fine-tuned model for re-use

If you want to keep the seed-42 model checkpoint for later evaluation (e.g., on a different paraphrase track or domain), uncomment and run the cell below. The checkpoint is ~500MB.

In [ ]:
# import shutil
# shutil.make_archive('roberta_seed42', 'zip', './roberta_seed42')
# from google.colab import files
# files.download('roberta_seed42.zip')